# v3b seed sweep (seeds 1 and 2) on Colab

Runs `scripts/run_v3b_study.sh` with `SEEDS="0 1 2" ROUNDS=5`. Seed-0 stages are skipped because their artefacts come from the bundle; seeds 1-2 run the 5-round aggregation loop and the 200 / 700 EE direct SUMO PPO arms, then the manifest, final evaluation on T and O and figures are rebuilt with all three seeds.

**Before running:** (1) push the repo (commit `a38da14` or later); (2) upload `runs/colab/v3b_seed0_bundle.zip` (231 MB: round-0 store, ensemble, seed-0 run dirs, eval JSONLs, ledgers, E0 / ALINEA reports) to `MyDrive/traffic-surrogate-rl-bundle/`.

Everything lives on Drive, so a dead session is resumed by re-running cells 1-3 and 5: finished aggregation rounds are kept (`RESUME=1` → `--resume`), unfinished direct arms restart from scratch. Budget: loops ≈ 6-8 h on 8 cores; each 700 EE direct arm ≈ 14 h of SUMO, so plan on 2-3 sessions (Colab Pro) — the driver is idempotent per stage.

In [ ]:
# 1. Drive + repo
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive
import os
if not os.path.isdir('traffic-surrogate-rl'):
    !git clone https://github.com/LejunZhou/traffic-surrogate-rl.git
%cd traffic-surrogate-rl
!git pull --ff-only
!git log --oneline -1

In [ ]:
# 2. SUMO + Python deps (eclipse-sumo wheel ships the binaries and tools)
!pip install -q eclipse-sumo traci sumolib
import sumo, os
os.environ['SUMO_HOME'] = os.path.dirname(sumo.__file__)
os.environ['PATH'] = os.path.join(os.environ['SUMO_HOME'], 'bin') + ':' + os.environ['PATH']
!sumo --version | head -1 && netconvert --version | head -1
!pip install -q -e . 2>&1 | tail -1
!nproc; import multiprocessing; print('workers:', multiprocessing.cpu_count())

In [ ]:
# 3. Seed-0 artefacts (once; the check skips if the store is already there)
import os
if not os.path.isfile('data/plant_v3b/round0/split_index.json'):
    !unzip -q -o /content/drive/MyDrive/traffic-surrogate-rl-bundle/v3b_seed0_bundle.zip -d .
!ls data/plant_v3b/round0 | wc -l; ls runs/surrogate/plant_v3b_r0 runs/aggregation/v3b_s0 | head; ls runs/study/v3b
# stale partial dirs from an earlier interrupted session are fine for the loops (RESUME=1) but not for the
# direct arms: remove a direct_ppo_*_s{1,2} dir (and its runs/ledger/v3b_direct_*_s{1,2}.jsonl) that has no final_model.zip
!for d in runs/study/v3b/direct_ppo_*_s1 runs/study/v3b/direct_ppo_*_s2; do [ -d "$d" ] && [ ! -f "$d/final_model.zip" ] && echo "stale: $d"; done; true

In [ ]:
# 4. (optional) smoke test of the tool chain, ~1 min: one SUMO episode through the env
!PYTHONPATH=src python -c "import numpy as np; from utils.config import load_config; from rl.sumo_env_wrapper import SumoEnv; from sumo_env.demand_profiles import DemandProfile; cfg=dict(load_config('configs/experiments/round0_v3b.yaml')['env']); cfg['project_root']='.'; cfg.pop('density_stats_from',None); cfg['network_dir']='/tmp/net_smoke'; e=SumoEnv(cfg); e.reset(options={'profile': DemandProfile.constant(1800.0,600.0)}); r=0
for _ in range(120):
    o,rew,t,tr,i=e.step(np.array([0.5],dtype=np.float32)); r+=rew
print('episode return', round(float(r),1), 'teleports', i.get('teleports')); e.close()"

In [ ]:
# 5. Launch (detached from the cell; re-run this cell after a session restart — every stage is idempotent)
import multiprocessing, subprocess, os
env = dict(os.environ, SEEDS='0 1 2', ROUNDS='5', RESUME='1', WORKERS=str(multiprocessing.cpu_count()))
os.makedirs('runs/logs', exist_ok=True)
log = open('runs/logs/v3b_driver_colab.log', 'a')
p = subprocess.Popen(['sh', 'scripts/run_v3b_study.sh'], env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
open('runs/logs/v3b_driver_colab.pid', 'w').write(str(p.pid)); print('driver pid', p.pid)

In [ ]:
# 6. Status (re-run any time)
!tail -3 runs/logs/v3b_driver_colab.log
!for s in 1 2; do echo "--- loop s$s"; grep -h '^=====\|SUMO V\|stop' runs/logs/v3b_agg_s$s.log 2>/dev/null | tail -3; done
!for f in runs/logs/v3b_direct_*_s1.log runs/logs/v3b_direct_*_s2.log; do echo "--- $f: $(grep -h total_timesteps $f 2>/dev/null | tail -1)"; done
!ls runs/aggregation/*/study.json runs/study/v3b/direct_ppo_*/final_model.zip 2>/dev/null
!grep -l Traceback runs/logs/v3b_*.log 2>/dev/null; true

In [ ]:
# 7. When the driver log says 'study v3b complete': the numbers
!PYTHONPATH=src python scripts/run_final_evaluation.py --arms runs/study/v3b/arms.json --workers 2 --study v3b_final --sets test ood 2>&1 | tail -30
!ls _progress/figures/m14_v3b/